In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sumotosima/sumotoshima_processed/val/metadata_val.jsonl
/kaggle/input/sumotosima/sumotoshima_processed/val/images/AcuteOtitisMedia_77.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/ChronicOtitisMedia_24.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/CerumenImpaction_89.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/ChronicOtitisMedia_37.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/Myringosclerosis_22.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/ChronicOtitisMedia_73.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/AcuteOtitisMedia_33.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/Normal_44.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/Myringosclerosis_55.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/Normal_50.jpg
/kaggle/input/sumotosima/sumotoshima_processed/val/images/Myringosclerosis_16.jpg
/kaggle/input/sumotosima/sumotoshima_proce

In [2]:
# !pip install -q transformers sentence-transformers timm faiss-gpu

!pip install -q transformers sentence-transformers timm

In [3]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoTokenizer, AutoModel

# ==========================================
# 1. CẤU HÌNH DATASET (Cập nhật theo JSONL của bạn)
# ==========================================
class ENTREPJsonDataset(Dataset):
    def __init__(self, metadata_file, image_dir, transform=None):
        """
        Args:
            metadata_file: Đường dẫn file .jsonl
            image_dir: Thư mục chứa ảnh
            transform: Các bước augmentation
        """
        self.image_dir = image_dir
        self.transform = transform
        self.data = self._load_jsonl(metadata_file)

    def _load_jsonl(self, filepath):
        data = []
        # Đếm số file không tìm thấy để debug
        missing_files = 0

        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    entry = json.loads(line)
                    # Mapping theo dữ liệu bạn cung cấp:
                    # {"image": "...", "text": "...", "label": 1}
                    img_name = entry.get('image')
                    text = entry.get('text')

                    if img_name and text:
                        full_path = os.path.join(self.image_dir, img_name)

                        # Kiểm tra file có tồn tại không để tránh lỗi lúc train
                        if os.path.exists(full_path):
                            data.append({
                                "image_path": full_path,
                                "text": str(text)
                            })
                        else:
                            missing_files += 1
                except ValueError:
                    continue

        print(f"--- DATASET INFO ---")
        print(f"Total entries loaded: {len(data)}")
        print(f"Missing images skipped: {missing_files}")
        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load ảnh (Convert RGB để tránh lỗi kênh alpha)
        try:
            image = Image.open(item['image_path']).convert('RGB')
        except Exception as e:
            # Fallback nếu ảnh lỗi
            print(f"Corrupt image: {item['image_path']}")
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "text": item['text']
        }

# ==========================================
# 1. CẤU HÌNH TRANSFORMS
# ==========================================

# Train: Có Augmentation (lật, xoay, chỉnh màu) để chống Overfit
# train_transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.RandomCrop(224),
#     transforms.RandomHorizontalFlip(p=0.3),
#     transforms.RandomRotation(degrees=10),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# # Valid: CHỈ Resize và Normalize (Giữ nguyên ảnh gốc để đánh giá chuẩn)
# val_transform = transforms.Compose([
#     transforms.Resize((224, 224)), # Resize thẳng về 224
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # chỉ resize, không crop
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [4]:
# ==========================================
# 2. MÔ HÌNH NANOCLIP (Bi-Encoder)
# ==========================================
class NanoCLIP(nn.Module):
    def __init__(self):
        super(NanoCLIP, self).__init__()

        # IMAGE ENCODER: DINOv2 ViT-S/14 [cite: 135]
        self.image_encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

        # Freeze toàn bộ, chỉ train 4 block cuối + norm
        for param in self.image_encoder.parameters():
            param.requires_grad = False
        for param in self.image_encoder.blocks[-4:].parameters():
            param.requires_grad = True
        self.image_encoder.norm.requires_grad = True

        # TEXT ENCODER: all-MiniLM-L6-v2 [cite: 137]
        self.text_encoder = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
        self.tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

        # Freeze toàn bộ, chỉ train 4 layer cuối
        for param in self.text_encoder.embeddings.parameters():
            param.requires_grad = False
        for layer in self.text_encoder.encoder.layer[:-4]:
             for param in layer.parameters():
                param.requires_grad = False

        # PROJECTION LAYERS (384 -> 64) [cite: 136, 138]
        self.img_proj = nn.Linear(384, 64)
        self.txt_proj = nn.Linear(384, 64)

        # Temperature (Logit scale)
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.05))

    def encode_image(self, image):
        features = self.image_encoder(image) # (Batch, 384)
        embeddings = self.img_proj(features)
        return torch.nn.functional.normalize(embeddings, dim=1)

    def encode_text(self, text_list, device):
        inputs = self.tokenizer(text_list, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
        outputs = self.text_encoder(**inputs)

        # Mean Pooling
        attention_mask = inputs['attention_mask']
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        embeddings = self.txt_proj(embeddings)
        return torch.nn.functional.normalize(embeddings, dim=1)

    def forward(self, image, text_list):
        img_emb = self.encode_image(image)
        txt_emb = self.encode_text(text_list, image.device)
        return img_emb, txt_emb, self.logit_scale.exp()

In [5]:
# ==========================================
# 2. HÀM VALIDATION
# ==========================================
def validate(model, val_loader, criterion, device):
    model.eval() # Chuyển sang chế độ đánh giá (tắt Dropout, Batchnorm fix)
    total_val_loss = 0.0

    with torch.no_grad(): # Tắt tính toán gradient để tiết kiệm bộ nhớ
        for batch in val_loader:
            images = batch['image'].to(device)
            texts = batch['text']

            # Forward pass
            img_emb, txt_emb, scale = model(images, texts)

            # Tính Loss
            logits = scale * (img_emb @ txt_emb.t())
            labels = torch.arange(len(logits), device=device)

            loss_i2t = criterion(logits, labels)
            loss_t2i = criterion(logits.t(), labels)
            loss = (loss_i2t + loss_t2i) / 2

            total_val_loss += loss.item()

    avg_loss = total_val_loss / len(val_loader)
    return avg_loss

# ==========================================
# 3. TRAINING LOOP VỚI VALIDATION
# ==========================================
def train_pipeline_with_val():
    # --- PATHS (Bạn cần kiểm tra lại tên file jsonl của tập Valid) ---
    TRAIN_META = "/kaggle/input/sumotosima/sumotoshima_processed/train/metadata_train.jsonl"
    TRAIN_IMG = "/kaggle/input/sumotosima/sumotoshima_processed/train/images"

    # GIẢ ĐỊNH: Tập valid nằm ở folder 'val' hoặc 'validation'.
    # Hãy sửa lại đường dẫn này cho khớp với dataset của bạn trên Kaggle
    VAL_META = "/kaggle/input/sumotosima/sumotoshima_processed/val/metadata_val.jsonl"
    VAL_IMG = "/kaggle/input/sumotosima/sumotoshima_processed/val/images"

    # HYPERPARAMETERS
    BATCH_SIZE = 64
    LR = 5e-5
    EPOCHS = 10
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Load Datasets
    print("--- Loading Datasets ---")
    train_dataset = ENTREPJsonDataset(TRAIN_META, TRAIN_IMG, transform=transform)
    val_dataset = ENTREPJsonDataset(VAL_META, VAL_IMG, transform=transform)


    if len(train_dataset) == 0 or len(val_dataset) == 0:
        print("Lỗi: Train hoặc Val dataset bị rỗng. Kiểm tra lại đường dẫn.")
        return

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # 2. Init Model
    model = NanoCLIP().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 8], gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    # Biến để theo dõi Best Model
    best_val_loss = float('inf')

    print(f"--- Start Training on {DEVICE} (Train: {len(train_dataset)}, Val: {len(val_dataset)}) ---")

    for epoch in range(EPOCHS):
        # === TRAIN PHASE ===
        model.train()
        train_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            images = batch['image'].to(DEVICE)
            texts = batch['text']

            optimizer.zero_grad()
            img_emb, txt_emb, scale = model(images, texts)

            logits = scale * (img_emb @ txt_emb.t())
            labels = torch.arange(len(logits), device=DEVICE)

            loss = (criterion(logits, labels) + criterion(logits.t(), labels)) / 2

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"[Epoch {epoch+1}] Step {batch_idx}: Train Loss = {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)

        # === VALIDATION PHASE ===
        print(f"Evaluating Epoch {epoch+1}...")
        avg_val_loss = validate(model, val_loader, criterion, DEVICE)

        # Scheduler Step
        scheduler.step()

        # === LOGGING & SAVING ===
        print(f"=== EPOCH {epoch+1} DONE ===")
        print(f"    Train Loss: {avg_train_loss:.4f}")
        print(f"    Val Loss  : {avg_val_loss:.4f}")

        # Lưu checkpoint cho epoch hiện tại
        torch.save(model.state_dict(), f"nanoclip_last.pth")

        # Lưu Best Model
        if avg_val_loss < best_val_loss:
            print(f"    => New Best Model! (Val Loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f})")
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "nanoclip_best.pth")
        else:
            print(f"    => Val Loss did not improve.")

if __name__ == "__main__":
    # Đảm bảo bạn đã định nghĩa class ENTREPJsonDataset và NanoCLIP từ trước
    train_pipeline_with_val()

--- Loading Datasets ---
--- DATASET INFO ---
Total entries loaded: 400
Missing images skipped: 0
--- DATASET INFO ---
Total entries loaded: 50
Missing images skipped: 0
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 308MB/s]


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

=== EPOCH 9 DONE ===
    Train Loss: 2.2636
    Val Loss  : 2.4488
    => New Best Model! (Val Loss improved from 2.4642 to 2.4488)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Epoch 10] Step 0: Train Loss = 2.3450
Evaluating Epoch 10...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


=== EPOCH 10 DONE ===
    Train Loss: 2.1947
    Val Loss  : 2.4415
    => New Best Model! (Val Loss improved from 2.4488 to 2.4415)


In [7]:
import torch
import numpy as np
import json
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

# --- 1. HÀM TÍNH TOÁN METRICS (Recall, Precision, MRR, nDCG, mAP) ---
# Dựa trên định nghĩa trong bài báo [cite: 111-126]
def calculate_metrics(sim_matrix, k_values=[1, 5, 10]):
    """
    sim_matrix: (N_images x N_texts) - Hàng là ảnh, Cột là text.
    Giả định: Text thứ i khớp với Ảnh thứ i (Ground truth nằm trên đường chéo).
    """
    # Chuyển vị để hàng là Query (Text), cột là Target (Image)
    scores = sim_matrix.T
    n_queries = scores.shape[0]

    # Ground truth là đường chéo (Text i khớp Image i)
    targets = np.arange(n_queries)

    # Sắp xếp giảm dần điểm số để lấy thứ hạng
    sorted_indices = np.argsort(-scores, axis=1)

    metrics = {
        "Recall": {},
        "Precision": {},
        "nDCG": {},
        "MRR": 0.0,
        "mAP": 0.0
    }

    reciprocal_ranks = []

    # --- Tính MRR & mAP ---
    # (Với 1 ground truth duy nhất, mAP chính là MRR)
    for i in range(n_queries):
        # Tìm vị trí (rank) của ảnh đúng
        # np.where trả về index trong mảng đã sort (bắt đầu từ 0)
        rank_pos = np.where(sorted_indices[i] == targets[i])[0][0]
        reciprocal_ranks.append(1.0 / (rank_pos + 1))

    metrics["MRR"] = np.mean(reciprocal_ranks)
    metrics["mAP"] = np.mean(reciprocal_ranks) # Tương đương MRR khi chỉ có 1 ground truth

    # --- Tính Metrics @K ---
    for k in k_values:
        recalls = []
        precisions = []
        ndcgs = []

        # Chỉ lấy Top-K dự đoán
        top_k_indices = sorted_indices[:, :k]

        for i in range(n_queries):
            target_idx = targets[i]

            if target_idx in top_k_indices[i]:
                # 1. Recall@K: Tìm thấy = 1, Không = 0
                recalls.append(1.0)

                # 2. Precision@K: (Số ảnh đúng tìm thấy) / K
                # Vì chỉ có 1 ảnh đúng, nếu tìm thấy thì là 1/K
                precisions.append(1.0 / k)

                # 3. nDCG@K
                # IDCG = 1 (vì ảnh đúng ở top 1 có rel=1)
                # DCG = 1 / log2(rank + 1)
                rank_in_top_k = np.where(top_k_indices[i] == target_idx)[0][0] + 1
                ndcg = 1.0 / np.log2(rank_in_top_k + 1)
                ndcgs.append(ndcg)
            else:
                recalls.append(0.0)
                precisions.append(0.0)
                ndcgs.append(0.0)

        metrics["Recall"][f"@{k}"] = np.mean(recalls)
        metrics["Precision"][f"@{k}"] = np.mean(precisions)
        metrics["nDCG"][f"@{k}"] = np.mean(ndcgs)

    return metrics

# --- 2. HÀM TRÍCH XUẤT EMBEDDINGS ---
def get_all_embeddings(model, loader, device):
    """
    Chạy model qua toàn bộ dataset để lấy vector đặc trưng.
    """
    model.eval()
    all_img_embs = []
    all_txt_embs = []

    print("Extracting embeddings...")
    with torch.no_grad():
        for batch in tqdm(loader):
            images = batch['image'].to(device)
            texts = batch['text']

            # Encode riêng lẻ
            img_emb = model.encode_image(images)
            txt_emb = model.encode_text(texts, device)

            all_img_embs.append(img_emb.cpu().numpy())
            all_txt_embs.append(txt_emb.cpu().numpy())

    # Nối lại thành 1 ma trận lớn (N_samples, 64)
    all_img_embs = np.concatenate(all_img_embs, axis=0)
    all_txt_embs = np.concatenate(all_txt_embs, axis=0)

    return all_img_embs, all_txt_embs

# --- 3. MAIN TEST SCRIPT ---
def run_test_evaluation():
    # --- PATHS (Cấu hình đường dẫn Test Set) ---
    TEST_META = "/kaggle/input/sumotosima/sumotoshima_processed/test/metadata_test.jsonl"
    TEST_IMG = "/kaggle/input/sumotosima/sumotoshima_processed/test/images"
    MODEL_PATH = "nanoclip_best.pth" # Load model tốt nhất vừa train

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BATCH_SIZE = 64

    # 1. Load Data (Dùng val_transform: Chỉ Resize + Normalize, KHÔNG Augment)
    print("--- Loading Test Data ---")
    # Lưu ý: Cần đảm bảo class ENTREPJsonDataset và val_transform đã được định nghĩa ở cell trước
    test_dataset = ENTREPJsonDataset(TEST_META, TEST_IMG, transform=transform)

    if len(test_dataset) == 0:
        print("Test dataset is empty!")
        return

    # Shuffle=False để giữ thứ tự tương ứng (Text i ứng với Image i)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # 2. Load Model
    print(f"--- Loading Model from {MODEL_PATH} ---")
    model = NanoCLIP().to(DEVICE)

    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    else:
        print(f"Error: Model file '{MODEL_PATH}' not found. Using random weights (Just for debug).")

    # 3. Get Embeddings
    img_embs, txt_embs = get_all_embeddings(model, test_loader, DEVICE)
    print(f"Embeddings shape: Image {img_embs.shape}, Text {txt_embs.shape}")

    # 4. Calculate Similarity Matrix
    # (N, 64) @ (64, N) -> (N, N)
    print("Calculating Similarity Matrix...")
    sim_matrix = img_embs @ txt_embs.T

    # 5. Compute Metrics
    print("Computing Metrics...")
    results = calculate_metrics(sim_matrix, k_values=[1, 5, 10])

    # 6. Print Report
    print("\n" + "="*40)
    print("   EVALUATION REPORT ON TEST SET")
    print("="*40)
    print(f"Total Queries: {img_embs.shape[0]}")
    print("-" * 30)
    print(f"MRR       : {results['MRR']:.4f}")
    print(f"mAP       : {results['mAP']:.4f}")
    print("-" * 30)

    for k in [1, 5, 10]:
        print(f"Recall@{k:<2}   : {results['Recall'][f'@{k}']:.4f}")
        print(f"Precision@{k:<2}: {results['Precision'][f'@{k}']:.4f}")
        print(f"nDCG@{k:<2}     : {results['nDCG'][f'@{k}']:.4f}")
        print("-" * 30)

if __name__ == "__main__":
    run_test_evaluation()

--- Loading Test Data ---
--- DATASET INFO ---
Total entries loaded: 50
Missing images skipped: 0
--- Loading Model from nanoclip_best.pth ---


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Extracting embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 1/1 [00:00<00:00,  1.22it/s]

Embeddings shape: Image (50, 64), Text (50, 64)
Calculating Similarity Matrix...
Computing Metrics...

   EVALUATION REPORT ON TEST SET
Total Queries: 50
------------------------------
MRR       : 0.3392
mAP       : 0.3392
------------------------------
Recall@1    : 0.1600
Precision@1 : 0.1600
nDCG@1      : 0.1600
------------------------------
Recall@5    : 0.5200
Precision@5 : 0.1040
nDCG@5      : 0.3371
------------------------------
Recall@10   : 0.9600
Precision@10: 0.0960
nDCG@10     : 0.4792
------------------------------
